# Inference Time Study

In [ ]:
################################################## Initialize ##################################################

# Add the new path
import sys
new_path = "/home/michele/code/michele_mmdet3d/"
if not new_path in sys.path:
    sys.path.insert(1, new_path)

# Main variables
    # Boolean to print inputs/outputs of each stage
wanna_print_in_out = True
    # Home directory within ADE
home_dir = '/home/michele/code/'
    # Relative path from "home_dir" to the pointcloud and image files
path_from_home_to_pointcloud_files = "michele_mmdet3d/data/minerva_polimove/training/velodyne_reduced"
path_from_home_to_image_files = "michele_mmdet3d/data/minerva_polimove/training/image_2"
    # Path to the ".txt" file in "ImageSets", containing the list of validation files
val_list_txt_file = "/home/michele/code/michele_mmdet3d/data/minerva_polimove/ImageSets/val.txt"
    # Built-in inferencer boolean, to select the type of built model
boolean_built_in_inferencer = False



############################################# Time-related variables #############################################
import time
import torch
if not torch.cuda.is_available():
    raise MemoryError("\t\tCUDA not available: exiting...\n\n\n")

delta_preprocessing = []
delta_inference = []
delta_postprocessing = []



################################################ Create the model ################################################
from mmdet3d.apis.inferencers import MultiModalityDet3DInferencer

# Build the model with the inferencer
if not boolean_built_in_inferencer:
    inferencer = MultiModalityDet3DInferencer(model="/home/michele/code/michele_mmdet3d/configs/minerva/MINERVA_mvxnet.py",
                                            weights="/home/michele/code/michele_mmdet3d/work_dirs/MINERVA_mvxnet/epoch_50.pth")
    model_cfg = inferencer.cfg
    model = inferencer.model



############################################### Automatic configs ###############################################
config_dictionary = {}

# For the loader LoadPointsFromFile
config_dictionary['LoadPointsFromFile'] = {
    'coord_type': model_cfg.val_dataloader.dataset.pipeline[0].coord_type,
    'load_dim': model_cfg.val_dataloader.dataset.pipeline[0].load_dim,
    'use_dim': model_cfg.val_dataloader.dataset.pipeline[0].use_dim
}

config_dictionary['Det3DDataPreprocessor'] = {
    'voxel': model_cfg.model.data_preprocessor.voxel,
    'voxel_type': model_cfg.model.data_preprocessor.voxel_type,
    'voxel_layer': model_cfg.model.data_preprocessor.voxel_layer,
    'mean': model_cfg.model.data_preprocessor.mean,
    'std': model_cfg.model.data_preprocessor.std,
    'bgr_to_rgb': model_cfg.model.data_preprocessor.bgr_to_rgb,
    'pad_size_divisor': model_cfg.model.data_preprocessor.pad_size_divisor
}

In [ ]:
############################################### Very STARTING input ###############################################

# Read the names of the validation files from the ".txt" file in "ImageSets"
with open(val_list_txt_file, 'r') as file:
    val_file_names = sorted([line.strip() for line in file])

# Create the inputs
#   - See the details in "inference_and_model_FUSION.ipynb"
#   - For each input path to Pointcloud-Image-InfoFile
import os
inputs = []
for file_name in val_file_names:
    inputs.append([
        os.path.join(home_dir, path_from_home_to_pointcloud_files, file_name+".bin"),
        os.path.join(home_dir, path_from_home_to_image_files, file_name+".png")
    ])

# Print the output of this stage
if wanna_print_in_out:
    print("\nStarting values:")
    for element in inputs:
        print(element)

In [ ]:
###################################################################################################################
###############################################  LoadPointsFromFile ###############################################
###############################################                     ###############################################
###############################################      ...and...      ###############################################
###############################################                     ###############################################
###############################################  LoadImageFromFile  ###############################################
###################################################################################################################

from mmdet3d.datasets.transforms.loading import LoadPointsFromFile
from mmcv.transforms.loading import LoadImageFromFile

# Initialize the loaders
loader_pointcloud = LoadPointsFromFile(
    coord_type=config_dictionary['LoadPointsFromFile']['coord_type'],
    load_dim=config_dictionary['LoadPointsFromFile']['load_dim'],
    use_dim=config_dictionary['LoadPointsFromFile']['use_dim']
)
loader_image = LoadImageFromFile()

print(config_dictionary['LoadPointsFromFile']['coord_type'])

# Print the input to this stage
if wanna_print_in_out:
    print("\nLoadPointsFromFile and LoadImageFromFile input:")
    for element in inputs:
        print(element)

# For cycle to also handle lists of inputs
for i in range(len(inputs)):
    
    # Prepare the string input for the loaders
    #   - The Pointcloud Loader needs two dictionaries, one nested into the other
    #   - The Image Loader needs just one dictionary
    inputs[i] = dict(
        lidar_points=dict(
            lidar_path=inputs[i][0]
        ),
        img_path=inputs[i][1]
    )

    # Actual modification of the dictionary
    loader_pointcloud(inputs[i])
    loader_image(inputs[i])

# Print the output of this stage
if wanna_print_in_out: 
    print("\nLoadPointsFromFile and LoadImageFromFile output:")
    for element in inputs:
        print(element)

In [ ]:
############################################### Pack3DDetInputs ###############################################

from mmdet3d.datasets.transforms.formating import Pack3DDetInputs

# Initialize the packer
packer = Pack3DDetInputs(keys=['points', 'img'])

# Print the input to this stage
if wanna_print_in_out: 
    print("\nPack3DDetInputs input:")
    for element in inputs:
        print(element)

# For cycle to also handle lists of inputs
for i in range(len(inputs)):
    
    # Actual modification of the dictionary
    packer(inputs[i])

# Print the output of this stage
if wanna_print_in_out: 
    print("\nPack3DDetInputs output:")
    for element in inputs:
        print(element)

In [ ]:
############################################### Det3DDataPreprocessor ###############################################

from mmdet3d.models.data_preprocessors.data_preprocessor import Det3DDataPreprocessor

# Initialize the preprocessor
preprocessor = Det3DDataPreprocessor(
    voxel=config_dictionary['Det3DDataPreprocessor']['voxel'],
    voxel_type=config_dictionary['Det3DDataPreprocessor']['voxel_type'],
    voxel_layer=config_dictionary['Det3DDataPreprocessor']['voxel_layer'],
    mean=config_dictionary['Det3DDataPreprocessor']['mean'],
    std=config_dictionary['Det3DDataPreprocessor']['std'],
    bgr_to_rgb=config_dictionary['Det3DDataPreprocessor']['bgr_to_rgb'],
    pad_size_divisor=config_dictionary['Det3DDataPreprocessor']['pad_size_divisor']
)

# Print the input to this stage
if wanna_print_in_out: 
    print("\nDet3DDataPreprocessor input:")
    for element in inputs:
        print(element)

# Create the list with the final inputs
final_inputs = []
for i in range(len(inputs)):    

    # Create a temporary dictionary to be passed to the preprocessor (in the right format)
    temp = dict(
        inputs=dict(
            points=[inputs[i]['points']],   ## Must be into a list, otherwise error
            img=[inputs[i]['img']])         ## Must be into a list, otherwise error
    )

    # Take out the result of the Det3DDataPreprocessor, and also compute the time
    start_preprocessing = time.time()
    final_inputs.append(
        preprocessor(temp)
    )
    torch.cuda.synchronize()
    end_preprocessing = time.time()

    # Append the time to the right list
    delta_preprocessing.append(end_preprocessing-start_preprocessing)

# Print the output of this stage
if wanna_print_in_out: 
    print("\nDet3DDataPreprocessor output:")
    for element in final_inputs:
        print(element)

In [ ]:
################################################# Inference #################################################

raw_results=[]
for element in final_inputs:
    # Take out the result of the inferencer, and also compute the time
    start_inference = time.time()
    raw_results.append(
        model(element['inputs'], mode='tensor')
    )
    torch.cuda.synchronize()
    end_inference = time.time()
    # Append the time to the right list
    delta_inference.append(end_inference-start_inference)

In [ ]:
############################################## Post_processing ##############################################

predictions=[]
for element in raw_results:
    # Take out the prediction, and also compute the time
    start_postprocessing = time.time()
    predictions.append(
        bbox_head.predict_by_feat(
            cls_scores = element['cls_score'],
            bbox_preds = element['bbox_pred'],
            dir_cls_preds = element['dir_cls_pred'],
            batch_input_metas = [metas_base_variable],
            cfg = test_cfg
        )
    )
    torch.cuda.synchronize()
    end_postprocessing = time.time()
    # Append the time to the right list
    delta_postprocessing.append(end_postprocessing-start_postprocessing)

In [8]:
#################################################################################################################
#                                            PLOTTING OF FREQUENCY                                              #
#################################################################################################################

from demo.plotters import *

# Check that the vectors are all of the same dimension
if len(delta_preprocessing) != len(delta_postprocessing) or len(delta_preprocessing) != len(delta_inference):
    print("\nWrong dimensions for lists!!!\n")
    exit()

# Create the vector with the TOTAL delta_time
delta_total = []
for i in range(len(delta_preprocessing)):
    delta_total.append(delta_preprocessing[i]+delta_inference[i]+delta_postprocessing[i])

In [ ]:
# Plot the total time

freq_plot_with_gaussian(delta_total, "Total time", "blue")

In [ ]:
# Plot the pre-processing time

freq_plot_with_gaussian(delta_preprocessing, "Pre-processing time", "green")

In [ ]:
#Plot the inference time

freq_plot_with_gaussian(delta_inference, "Network Inference time", "gold")

In [ ]:
# Plot the post-processing time

freq_plot_with_gaussian(delta_postprocessing, "Post-processing time", "black", 20)

In [ ]:
# Plot the pie chart with the percentages

plot_pie_chart(delta_preprocessing, delta_inference, delta_postprocessing)